In [ ]:
import sys; sys.path.append('..')
sys.path.append('../curved_linesearch/')
import MeshFEM, mesh, mesh_energy, benchmark, viewer, py_newton_optimizer

import numpy as np
import igl

import matplotlib
from matplotlib import pyplot as plt

In [ ]:
import sim_utils, param_utils
import extra_utils, opt_utils

In [ ]:
from curved_linesearch import visualization

In [ ]:
import newton_flow
import newton_flow_utils as nfu

# Load Optimal Layout and Initial Emebedding

## Param 

In [ ]:
m_rest = mesh.Mesh('ToysMesh/Hilbert_opt_2d.obj', embeddingDimension=3)
m_rest_2 = mesh.Mesh('ToysMesh/Hilbert_opt_2d.obj')
m_defo = mesh.Mesh('ToysMesh/Hilbert_init_2d.obj')

In [ ]:
FIX_VARS = False
always_project = False

In [ ]:
param_poisson, prob_poisson, uv_vars = extra_utils.getParamProb(m_rest, m_defo.vertices(), FIX_VARS=FIX_VARS)

In [ ]:
param_poisson.elementHessianShift = 1e-8
prob_poisson.hessianShift = 0
prob_poisson.useRelativeHessianShift = False

In [ ]:
opt = prob_poisson.optimizer()
opt.options.hessianProjectionController.startWithProjectionActive = False
opt.options.hessianProjectionController.numProjectionStepsBeforeDisable = 1
opt.options.hessianProjectionController.numConsecutiveIndefiniteStepsBeforeEnable = 0
if always_project: opt.options.hessianProjectionController = py_newton_optimizer.HessianProjectionAlways()

In [ ]:
prob_poisson.energy()

## Newton Flow

In [ ]:
v = mesh_energy.NodalVars(m_rest_2, 2)
v.setVars(m_defo.vertices().ravel())

In [ ]:
nf = newton_flow.symmetric_dirichlet(m_rest_2, v)
prob = py_newton_optimizer.NewtonMultiobjectiveProblem(v, [nf])

In [ ]:
opt_2 = prob.optimizer()
opt.options.niter = 200
opt_2.options = opt.options

In [ ]:
nf.elementHessianShift = param_poisson.elementHessianShift 
prob.hessianShift = prob_poisson.hessianShift 
prob.useRelativeHessianShift = prob_poisson.useRelativeHessianShift 

In [ ]:
prob.energy()

# Construct Extrapolate Method Class

In [ ]:
linear_extrapolator = extra_utils.LinearExtrapolator()

In [ ]:
pade_extrapolator = extra_utils.PadeExtrapolator(prob, opt_2, 14)

In [ ]:
RS_extrapolator = extra_utils.RotationStrainExtrapolation(prob_poisson)

In [ ]:
RS_extrapolator.G

In [ ]:
x0 = prob_poisson.getVars()

In [ ]:
d = opt.newton_step()

In [ ]:
RS_extrapolator.linesearch_begin(x0, d)

In [ ]:
RS_extrapolator.d_grad

In [ ]:
brek

# Newton Optimize

In [ ]:
line_search_method = opt_utils.BruteForceLinesearch(alpha_step_size=0.01)

In [ ]:
# opt_2.options.factorizer = opt_2.options.factorizer.CatamariAMD

In [ ]:
# benchmark.reset()
# opt.optimize()
# benchmark.report()


In [ ]:
benchmark.reset()
vertices_list = opt_utils.newton_extrapolate(opt, linear_extrapolator, line_search_method, max_iters=10, verbose=True)
benchmark.report()

In [ ]:
max_alpha = 10
line_search_method.alpha_step_size = 0.1
line_search_method.max_alpha = max_alpha

In [ ]:
benchmark.reset()
vertices_list = opt_utils.newton_extrapolate(opt, RS_extrapolator, line_search_method, max_iters=100, verbose=True)
benchmark.report()

In [ ]:
brek

In [ ]:
final_uv = vertices_list[-1]

In [ ]:
initial_uv = vertices_list[0]

# UV Viewer

In [ ]:
uv_final = mesh.Mesh(final_uv, m_rest.elements())

In [ ]:
uv_viewer = viewer.Viewer(uv_final, wireframe=True)
uv_viewer.show()

## Flow Visualization

In [ ]:
fv = vertices_list

In [ ]:
extrapolation_dist = max_alpha
constant_speed = False
num_frames = min(500, len(fv))

methods = [(1, nfu.eval_trajectory_taylor, 'Newton'),
           (2, nfu.eval_trajectory_taylor, 'Deg 2 Taylor'),
           (3, nfu.eval_trajectory_taylor, 'Deg 3 Taylor'),
           # (1, extra_utils.RotationStrainExtrapolation(prob_poisson), 'Poisson'),
           (14, nfu.eval_trajectory_vector_pade, 'Pade 14'),
           (19, nfu.eval_trajectory_vector_pade, 'Pade 19')
]




ff = lambda i:  visualization.flow_frame(i, opt_2, fv, extrapolation_dist, constant_speed,
                         extrapolation_method_list=methods, truncate=True, corners_only=True)

In [ ]:
# visualization.writeVideo('videos/pade_vis_traj.mp4',num_frames,ff)

In [ ]:
ff(28)